In [1]:
# Here, I will write the commands to infer eGRNs with Pando in chunk-wise.

In [2]:
getwd()

[1] "/fast/AG_Bunina/Yusuf/Project_Endothelial_and_Stroke/Datasets/Chromatin_and_Gene_Exp/2024_C_A_Mannens_C_et_al/04_02_25"

In [3]:
here::here()

[1] "/fast/AG_Bunina/Yusuf/Project_Endothelial_and_Stroke/Datasets/Chromatin_and_Gene_Exp/2024_C_A_Mannens_C_et_al/04_02_25"

In [4]:
# load the R environment with the necessary packages such as Epiregulon:

my_epiregulon_lib <- here::here("renv", "library/linux-rhel-9.4/R-4.4/x86_64-unknown-linux-gnu")

In [5]:
.libPaths(new = my_epiregulon_lib, include.site = FALSE)

In [6]:
.libPaths()

[1] "/fast/AG_Bunina/Yusuf/Project_Endothelial_and_Stroke/Datasets/Chromatin_and_Gene_Exp/2024_C_A_Mannens_C_et_al/04_02_25/renv/library/linux-rhel-9.4/R-4.4/x86_64-unknown-linux-gnu"
[2] "/gnu/store/29x2k7i71g9xq09xmbj1lk515cl7if63-r-minimal-4.4.2/lib/R/library"

In [7]:
library(magrittr)

In [8]:
here::here('r_objects') |> list.files() %>% print()

 [1] "GeneExpressionMatrix.RDS"                        
 [2] "GeneExpressionMatrix_used_for_epiregulon.RDS"    
 [3] "PeakMatrix.RDS"                                  
 [4] "PeakMatrix_used_for_epiregulon.RDS"              
 [5] "TF_activity_chunks"                              
 [6] "mannens_et_al_seurat_modified.RDS"               
 [7] "mannens_et_al_seurat_obj.RDS"                    
 [8] "metadata.RDS"                                    
 [9] "my_human_pwms_v2.RDS"                            
[10] "pando_eGRN_chunks"                               
[11] "pca_atac_by_bpcells_package.RDS"                 
[12] "pca_rna_by_bpcells_package.RDS"                  
[13] "peaks_granges_for_TF_footprinting_by_BPCells.RDS"
[14] "regulon_weighted_by_celltypes.RDS"               
[15] "sce_atac_mannens.RDS"                            
[16] "sce_rna_mannens.RDS"                             
[17] "tf_activity_by_all_cells_matrix_NORMALIZED.RDS"  
[18] "tf_activity_by_cells_matrix.RDS"          

In [9]:
mannens_seurat <- 
    readRDS(here::here('r_objects', 'mannens_et_al_seurat_modified.RDS'))

In [10]:
library(tidyverse)

-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.5.1
v ggplot2   3.5.1     v tibble    3.2.1
v lubridate 1.9.3     v tidyr     1.3.1
v purrr     1.0.4     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x tidyr::extract()   masks magrittr::extract()
x dplyr::filter()    masks stats::filter()
x dplyr::lag()       masks stats::lag()
x purrr::set_names() masks magrittr::set_names()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [11]:
library(Seurat)

Loading required package: SeuratObject

Loading required package: sp

'SeuratObject' was built under R 4.4.1 but the current version is
4.4.2; it is recomended that you reinstall 'SeuratObject' as the ABI
for R may have changed


Attaching package: 'SeuratObject'


The following objects are masked from 'package:base':

    intersect, t




In [12]:
library(Signac)

In [13]:
library(Pando)


Attaching package: 'Pando'


The following objects are masked from 'package:Seurat':

    GetAssay, VariableFeatures


The following objects are masked from 'package:SeuratObject':

    LayerData, VariableFeatures




In [14]:
library(doParallel)

Loading required package: foreach


Attaching package: 'foreach'


The following objects are masked from 'package:purrr':

    accumulate, when


Loading required package: iterators

Loading required package: parallel



In [15]:
library(GenomicRanges)

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: 'BiocGenerics'


The following object is masked from 'package:SeuratObject':

    intersect


The following objects are masked from 'package:lubridate':

    intersect, setdiff, union


The following objects are masked from 'package:dplyr':

    combine, intersect, setdiff, union


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    Filter, Find, Map, Position, Reduce, anyDuplicated, aperm, append,
    as.data.frame, basename, cbind, colnames, dirname, do.call,
    duplicated, eval, evalq, get, grep, grepl, intersect, is.unsorted,
    lapply, mapply, match, mget, order, paste, pmax, pmax.int, pmin,
    pmin.int, rank, rbind, rownames, sapply, setdiff, table, tapply,
    union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: 'S4Vectors'


The following o

In [16]:
# registerDoParallel(16) # it does not work properly in jupyter lab notebook !!!!

In [17]:
# Function to retrieve elements in chunks with a 5-gene overlap
retrieve_in_chunks <- function(gene_names, chunk_size, overlap = 5) {
  
  # Get the total number of gene names provided in the input
  n <- length(gene_names)  

  # Define the step size (how much to move forward after creating each chunk).
  # The step size is calculated by subtracting the overlap from the chunk size.
  # This ensures that each successive chunk shares 'overlap' number of genes with the previous chunk.
  step_size <- chunk_size - overlap  

  # Initialize an empty list to store the resulting chunks of gene names.
  chunk_list <- list()  

  # Use a for-loop to iterate through the gene names vector in steps of 'step_size'.
  # The 'seq()' function generates a sequence of starting points (i) for each chunk.
  # The loop increments by 'step_size' to determine the next chunk’s starting point.
  for (i in seq(1, n, by = step_size)) {
    
    # Extract the current chunk of gene names from the vector.
    # The 'min()' function ensures that the chunk doesn't go beyond the last gene in 'gene_names'.
    # The chunk starts at 'i' and ends at 'i + chunk_size - 1' (or the last available gene).
    chunk <- gene_names[i:min(i + chunk_size - 1, n)]
    
    # Generate a name for each chunk (optional, but helpful for referencing).
    # The 'ceiling()' function calculates which chunk we are in based on 'i' and 'step_size'.
    chunk_name <- paste0("chunk_", ceiling(i / step_size))
    
    # Add the current chunk to the 'chunk_list' under its generated name.
    # This allows for easy access to each chunk by its name.
    chunk_list[[chunk_name]] <- chunk
  }
  
  # Return the full list of chunks. Each element of the list corresponds to a chunk of gene names.
  return(chunk_list)  
}

In [18]:
mannens_seurat

An object of class Seurat 
430192 features across 49470 samples within 2 assays 
Active assay: RNA (25071 features, 5000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: peaks
 5 dimensional reductions calculated: pca, pca_atac, umap_rna, umap_atac, joint_umap

In [19]:
mannens_seurat %>% Features() %>% head()

[1] "MALAT1"   "AUTS2"    "NRXN1"    "MIR99AHG" "GRID2"    "NPAS3"

In [20]:
mannens_seurat %>% rownames() %>% head()

[1] "MALAT1"   "AUTS2"    "NRXN1"    "MIR99AHG" "GRID2"    "NPAS3"

In [21]:
mannens_seurat@assays$RNA@meta.data %>% dim()

[1] 25071    24

In [22]:
mannens_seurat@assays$RNA@meta.data %>% colnames()

[1] "_index"                              "Accession"                          
 [3] "Chromosome"                          "End"                                
 [5] "Gene"                                "NPeaks"                             
 [7] "Selected"                            "Start"                              
 [9] "Strand"                              "cv"                                 
[11] "mu"                                  "pos"                                
[13] "std"                                 "ensembl_IDs"                        
[15] "gene_biotype"                        "total_counts"                       
[17] "vf_vst_counts_mean"                  "vf_vst_counts_variance"             
[19] "vf_vst_counts_variance.expected"     "vf_vst_counts_variance.standardized"
[21] "vf_vst_counts_variable"              "vf_vst_counts_rank"                 
[23] "var.features"                        "var.features.rank"

In [23]:
identical(rownames(mannens_seurat), mannens_seurat@assays$RNA@meta.data$Gene)

[1] TRUE

In [24]:
mannens_seurat@assays$RNA@meta.data$Accession %>% head()

[1] "ENSG00000251562.8"  "ENSG00000158321.18" "ENSG00000179915.24"
[4] "ENSG00000215386.13" "ENSG00000152208.13" "ENSG00000151322.19"

In [25]:
mannens_seurat@assays$RNA@meta.data$Accession %>% duplicated() %>% table()

.
FALSE 
25071 

In [26]:
temp_seurat <- mannens_seurat

In [27]:
rownames(temp_seurat) <- temp_seurat@assays$RNA@meta.data$Accession

Warning message:
"Renaming features in v3/v4 assays is not supported"


In [28]:
temp_seurat %>% rownames() %>% head()

[1] "ENSG00000251562.8"  "ENSG00000158321.18" "ENSG00000179915.24"
[4] "ENSG00000215386.13" "ENSG00000152208.13" "ENSG00000151322.19"

In [29]:
temp_seurat %>% Features() %>% head()

[1] "ENSG00000251562.8"  "ENSG00000158321.18" "ENSG00000179915.24"
[4] "ENSG00000215386.13" "ENSG00000152208.13" "ENSG00000151322.19"

In [30]:
# Note: Features() returns rownames of Seurat object.

In [31]:
rm(temp_seurat)

In [32]:
Features(mannens_seurat) %>% head()

[1] "MALAT1"   "AUTS2"    "NRXN1"    "MIR99AHG" "GRID2"    "NPAS3"

In [33]:
mannens_seurat

An object of class Seurat 
430192 features across 49470 samples within 2 assays 
Active assay: RNA (25071 features, 5000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: peaks
 5 dimensional reductions calculated: pca, pca_atac, umap_rna, umap_atac, joint_umap

===============================================

import the mannens et al object with motifs:

===============================================

In [34]:
mannens_et_al_further_subsetted_w_Pando_motifs <- 
    readRDS(here::here('..', '05_09_24', 'R_Objects', 'mannens_et_al_further_subsetted_w_Pando_motifs.RDS'))

In [35]:
mannens_et_al_further_subsetted_w_Pando_motifs

An object of class "GRNData"
Slot "grn":
A RegulatoryNetwork object based on 1122 transcription factors


No network has been inferred

Slot "data":
An object of class Seurat 
430192 features across 49470 samples within 2 assays 
Active assay: RNA (25071 features, 5000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: peaks


In [36]:
mannens_et_al_further_subsetted_w_Pando_motifs@data

An object of class Seurat 
430192 features across 49470 samples within 2 assays 
Active assay: RNA (25071 features, 5000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: peaks

In [37]:
# test run:

In [38]:
# gene_names <- mannens_et_al_further_subsetted_w_Pando_motifs@data %>% rownames()

# gene_chunks <- retrieve_in_chunks(gene_names = gene_names[1:10], 
#                                   chunk_size = 3, # 500
#                                   overlap = 1)

In [39]:
# file_name <- substitute(mannens_et_al_further_subsetted_w_Pando_motifs) %>% as.character()  # retrieve object's name as character.   

#  output_file_name <- file_name %>% str_split_i(pattern = 'et_al', i = 1)

# for (i in 1:length(gene_chunks)) { 

#     gene_names <- gene_chunks[[i]]

#     mannens_seurat_w_eGRN <- infer_grn(
#                                       mannens_et_al_further_subsetted_w_Pando_motifs,
#                                       genes = gene_names,
#                                       peak_to_gene_method = 'GREAT',
#                                       parallel = T)

#     mannens_seurat_w_eGRN %>% 
#      saveRDS(here::here('r_objects', 'pando_eGRN_chunks',
#                          paste0(output_file_name, 'et_al_w_eGRNs_', 'SLURM', '_c', i , '.RDS')))

    
#     message("Saved R object for chunk ", i ,": FINISHED!")
    
#     rm(mannens_seurat_w_eGRN)
    
# }


# cat("\n STATUS: COMPLETED !!! \n")

In [41]:
# View eGRN generated for fist chunk of target genes:

# mannens_eGRNs_c1 <- 
#     readRDS(here::here('r_objects', 'pando_eGRN_chunks', 'mannens_et_al_w_eGRNs_SLURM_c1.RDS'))

In [42]:
# It worked, then I submitted a slurm job for all target genes. 

In [43]:
sessionInfo()

R version 4.4.2 (2024-10-31)
Platform: x86_64-unknown-linux-gnu
Running under: Red Hat Enterprise Linux 9.4 (Plow)

Matrix products: default
BLAS/LAPACK: /gnu/store/mj1kw87qd3m1q7r4844adkn5hifx8k6a-openblas-0.3.20/lib/libopenblasp-r0.3.20.so;  LAPACK version 3.9.0

locale:
 [1] LC_CTYPE=C          LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C.UTF-8
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] stats4    parallel  stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] GenomicRanges_1.56.2 GenomeInfoDb_1.40.1  IRanges_2.38.1      
 [4] S4Vectors_0.42.1     BiocGenerics_0.50.0  doParallel_1.0.17   
 [7] iterators_1.0.14     foreach_1.5.2        Pando_1.1.1         
[10] Signac_1.14.0        Seurat_5.2.1         SeuratObje